# AgentCore 셀프 매니지드 메모리 전략 데모

이 노트북은 boto3를 사용하여 Amazon Bedrock AgentCore 셀프 매니지드 메모리 전략을 설정하고 사용하는 방법을 보여줍니다. 셀프 매니지드 메모리 전략을 사용하면 대화 이벤트에 의해 트리거되는 메모리 추출 및 통합을 위한 커스텀 파이프라인을 만들 수 있습니다.

## 작동 방식

1. 트리거 설정: 단기 메모리 이벤트를 기반으로 파이프라인을 호출하는 트리거 조건(메시지 수, 유휴 타임아웃, 토큰 수)을 정의합니다
2. 알림 수신: 트리거 조건이 충족되면 AgentCore가 SNS 토픽에 알림을 게시합니다
3. 페이로드 처리: AgentCore가 S3 버킷에 대화 데이터를 전달합니다
4. 메모리 레코드 추출 및 저장: 커스텀 파이프라인이 페이로드를 검색하고 메모리를 처리합니다

셀프 매니지드 메모리 전략에 대한 자세한 정보는 [공식 AWS 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-self-managed-strategies.html#use-self-managed-strategy)를 참조하세요.

## 설정 개요

이 데모에서 수행할 작업:
1. 필요한 AWS 인프라(S3, SNS, SQS, Lambda, IAM 역할) 생성
2. 셀프 매니지드 전략으로 AgentCore 메모리 생성
3. 메모리 처리 파이프라인을 시연하기 위한 테스트 이벤트 생성
4. 저장된 메모리의 검색 및 사용을 시연하는 에이전트 생성
5. 완료 시 리소스 정리

## 설정 및 임포트

In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
import boto3
import json
import time
import uuid
import os
from datetime import datetime
from aws_utils import AWSUtils

# AWS 리전 설정
region_name = 'us-east-1'  # 원하는 리전으로 변경하세요
aws_utils = AWSUtils(region_name=region_name)

# Lambda 함수 코드 읽기
with open('lambda_function.py', 'r') as f:
    lambda_code = f.read()

## 1단계: 페이로드 전달을 위한 S3 버킷 생성

트리거 조건이 충족되면 AgentCore가 대화 페이로드를 전달할 S3 버킷을 생성합니다.

In [ ]:
# 고유한 이름으로 S3 버킷 생성
bucket_name = aws_utils.create_s3_bucket('agentcore-memory-payloads')
print(f"S3 버킷 생성 완료: {bucket_name}")

## 2단계: 메모리 작업 알림을 위한 SNS 토픽 생성

AgentCore가 메모리 처리 파이프라인을 트리거할 때 알림을 받을 SNS 토픽을 생성합니다.

In [ ]:
# SNS 토픽 생성
sns_topic_name = f"agentcore-memory-notifications-{int(time.time())}"
sns_topic_arn = aws_utils.create_sns_topic(sns_topic_name)
print(f"SNS 토픽 생성 완료: {sns_topic_arn}")

## 3단계: SNS 구독이 포함된 SQS 큐 생성

SNS 토픽을 구독하는 SQS 큐를 생성합니다. 이 큐는 Lambda 함수를 트리거할 메모리 작업 알림을 수신합니다.

In [ ]:
# SQS 큐를 생성하고 SNS 토픽을 구독합니다
queue_name = f"agentcore-memory-queue-{int(time.time())}"
queue_url, queue_arn = aws_utils.create_sqs_queue_with_sns_subscription(queue_name, sns_topic_arn)
print(f"SQS 큐 생성 완료: {queue_url}")

## 4단계: IAM 역할 생성

두 개의 IAM 역할을 생성합니다:
1. AgentCore가 S3 및 SNS에 접근하기 위한 역할
2. Lambda가 S3, SQS 및 AgentCore API에 접근하기 위한 역할

In [ ]:
# AgentCore용 IAM 역할 생성
agentcore_role_name = f"AgentCoreMemoryExecutionRole-{int(time.time())}"
agentcore_role_arn = aws_utils.create_iam_role_for_agentcore(
    agentcore_role_name, 
    bucket_name, 
    sns_topic_arn
)
print(f"AgentCore IAM 역할 생성 완료: {agentcore_role_arn}")

# Lambda용 IAM 역할 생성
lambda_role_name = f"LambdaMemoryProcessingRole-{int(time.time())}"
lambda_role_arn = aws_utils.create_iam_role_for_lambda(
    lambda_role_name, 
    bucket_name, 
    queue_arn
)
print(f"Lambda IAM 역할 생성 완료: {lambda_role_arn}")

## 5단계: 메모리 처리를 위한 Lambda 함수 생성

SQS 메시지에 의해 트리거될 Lambda 함수를 생성합니다. 이 함수는:
1. S3에서 대화 페이로드를 다운로드합니다
2. Bedrock 모델을 사용하여 메모리를 추출합니다
3. 추출된 메모리를 AgentCore에 다시 저장합니다

In [ ]:
# Lambda 함수 생성
function_name = f"agentcore-memory-processor-{int(time.time())}"
function_arn = aws_utils.create_lambda_function(
    function_name,
    lambda_role_arn,
    lambda_code
)
print(f"Lambda 함수 생성 완료: {function_arn}")

# Lambda에 SQS 트리거 추가
event_source_uuid = aws_utils.add_sqs_trigger_to_lambda(function_name, queue_arn)
print(f"Lambda에 SQS 트리거 추가 완료: {event_source_uuid}")

## 6단계: 셀프 매니지드 전략으로 AgentCore 메모리 생성

우리가 설정한 인프라를 사용하는 셀프 매니지드 전략 설정으로 AgentCore 메모리를 생성합니다.

In [ ]:

import importlib
import aws_utils
importlib.reload(aws_utils)

# 업데이트된 코드로 AWSUtils의 새 인스턴스를 생성합니다
aws_utils = aws_utils.AWSUtils(region_name=region_name)

# 셀프 매니지드 전략으로 메모리 생성
memory_name = f"SelfManageMemory{int(time.time())}"
memory_description = "Demo memory using self-managed strategy"

memory_id = aws_utils.create_memory_with_self_managed_strategy(
    memory_name=memory_name,
    memory_description=memory_description,
    role_arn=agentcore_role_arn,
    sns_topic_arn=sns_topic_arn,
    s3_bucket_name=bucket_name,
    message_trigger_count=3,  # 3개 메시지 후 트리거
    token_trigger_count=500,  # ~500 토큰 후 트리거
    idle_timeout=300,         # 5분 유휴 시간 후 트리거
    historical_window_size=5  # 컨텍스트에 이전 5개 메시지 포함
)

print(f"메모리 생성 완료: {memory_id}")
# print(f"Strategy ID: {strategy_id}")

In [ ]:
def wait_for_memory_to_get_active(memory_id):
    response = aws_utils.agentcore_client_control.get_memory(
        memoryId = memory_id)

    while response['memory']['status'] != 'ACTIVE':
        time.sleep(30)
        response = aws_utils.agentcore_client_control.get_memory(
        memoryId = memory_id)
        print(f"메모리 생성 상태: {response['memory']['status']}")
    return response['memory']['status']

wait_for_memory_to_get_active(memory_id=memory_id)

## 7단계: 메모리 파이프라인을 트리거하기 위한 테스트 이벤트 생성

이제 셀프 매니지드 메모리 파이프라인을 트리거하기 위한 테스트 이벤트를 생성합니다. 메시지 트리거 수를 초과할 만큼 충분한 이벤트를 생성합니다.

In [ ]:
actor_id = "test-user-123"

In [ ]:
# 테스트 이벤트 생성
session_id = aws_utils.create_test_events(
    memory_id=memory_id,
    actor_id=actor_id,
    num_events=6  # message_trigger_count 3을 초과합니다
)

print(f"세션 ID로 테스트 이벤트 생성 완료: {session_id}")

In [ ]:
aws_utils.agentcore_client.list_events(
    memoryId = memory_id, 
    actorId = actor_id,
    sessionId = session_id )

## 8단계: 메모리 처리 대기

이제 메모리 처리 파이프라인이 실행될 때까지 기다려야 합니다. 여기에는 다음이 포함됩니다:
1. AgentCore가 트리거 조건(메시지 수 초과)을 감지
2. AgentCore가 SNS에 알림을 게시
3. SNS가 SQS에 메시지를 전달
4. SQS가 Lambda 함수를 트리거
5. Lambda가 대화를 처리하고 메모리를 저장

잠시 기다린 후 메모리가 생성되었는지 확인합니다.

In [ ]:
print("메모리 처리 완료를 위해 30초 대기 중...")
time.sleep(30)

## 9단계: 메모리 레코드 확인

메모리 파이프라인이 메모리 레코드를 생성했는지 확인합니다.

In [ ]:
session_id

In [ ]:
# 메모리 레코드 목록 조회
namespace=f"/interests/actor/{actor_id}/session/{session_id}/"
def list_memory_records(memory_id, namespace):
    try:
        response = aws_utils.agentcore_client.list_memory_records(
            memoryId=memory_id,
            namespace=namespace
        )
        print(f"{len(response.get('memoryRecordSummaries'))}개의 메모리 레코드를 찾았습니다")
        
        # 검색 결과를 표시합니다
        for idx, result in enumerate(response.get("memoryRecordSummaries")):
            print(f"메모리: {idx}")
            print(f"내용: {result['content']['text']}")
    except Exception as e:
        print(f"메모리 검색 오류: {e}")
list_memory_records(memory_id, namespace)

위의 레코드에서 사용자 관심사의 반복이 보이는 것은 통합 로직을 추가하지 않았기 때문입니다. 따라서 반복이 있지만, 셀프 매니지드 전략을 제공할 수 있는 능력으로 추출과 수집만 원하는지 정의할 수 있습니다. 이는 비즈니스 사용 사례에 따라 달라집니다.

In [ ]:
# 메모리 레코드 검색
def retrieve_memory_records(memory_id, query, topK, namespace):
    try:
        response = aws_utils.agentcore_client.retrieve_memory_records(
            memoryId=memory_id,
            searchCriteria = {
            'searchQuery': query,
            'topK': topK
        },
            namespace=namespace
        )
        print(f"{len(response.get('memoryRecordSummaries'))}개의 메모리 레코드를 찾았습니다")
        
        # 검색 결과를 표시합니다
        for idx, result in enumerate(response.get('memoryRecordSummaries')):
            print(f"\n메모리 레코드 {idx + 1}:")
            print(f"내용: {result['content']['text']}")
    except Exception as e:
        print(f"메모리 검색 오류: {e}")

retrieve_memory_records(memory_id=memory_id, query="food choices for dinner", topK=5, namespace=namespace)

## 10단계: 다른 내용으로 추가 테스트 이벤트 생성

다른 내용으로 더 많은 테스트 이벤트를 생성하여 또 다른 메모리 처리 사이클을 트리거합니다.

In [ ]:
# 커스텀 테스트 이벤트 생성
session_id = str(uuid.uuid4())
actor_id = "test-user-456"

# 더 구체적인 정보가 포함된 커스텀 이벤트
test_events = [
    {
        "user": "I'm trying to eat healthier and have been exploring Mediterranean cuisine lately.",
        "assistant": "That's wonderful! Mediterranean food is both delicious and nutritious. What Mediterranean dishes have you tried so far?"
    },
    {
        "user": "I love Greek salads with feta cheese and olives, and I've been making homemade hummus.",
        "assistant": "Homemade hummus is fantastic! Do you prefer it with tahini or without? And what's your favorite way to serve it?"
    },
    {
        "user": "I always use tahini and like to serve it with fresh vegetables and pita bread. I'm also vegetarian, so I avoid meat.",
        "assistant": "Being vegetarian opens up so many Mediterranean options! Have you tried making stuffed grape leaves or lentil-based dishes?"
    },
    {
        "user": "Not yet, but I'd love to learn. I'm also allergic to shellfish, so I have to be careful with seafood dishes.",
        "assistant": "Good to know about the shellfish allergy. For vegetarian Mediterranean cooking, you might enjoy making moussaka with eggplant or trying some traditional Greek bean dishes. Would you like some recipe suggestions?"
    }
]

# 이벤트 생성
for idx, event in enumerate(test_events):
    try:
        event_payload = [
            {
                'conversational': {
                    'content': {
                        'text': event['user']
                    },
                    'role': 'USER'
                }
            },
            {
                'conversational': {
                    'content': {
                        'text': event['assistant']
                    },
                    'role': 'ASSISTANT'
                }
            }
        ]

        aws_utils.agentcore_client.create_event(
            memoryId=memory_id,
            actorId=actor_id,
            sessionId=session_id,
            eventTimestamp=int(time.time()),
            payload=event_payload,
            clientToken=str(uuid.uuid4())
        )

        print(f"이벤트 {idx+1}/{len(test_events)} 생성 완료")
        time.sleep(1)

    except Exception as e:
        print(f"테스트 이벤트 생성 오류: {e}")

print("\n메모리 처리 완료를 위해 30초 대기 중...")
time.sleep(30)

## 11단계: 새 메모리 검색

이제 하이킹과 사용자의 반려견에 관련된 새 메모리를 검색합니다.

In [ ]:
# 야외 활동에 대한 메모리 레코드 검색
namespace=f"/interests/actor/{actor_id}/session/{session_id}/"
retrieve_memory_records(memory_id=memory_id, query="dog pets golden retriever", topK=5, namespace=namespace)

## 12단계: 에이전트 생성

이 섹션에서는 훅을 통해 AgentCore 셀프 매니지드 메모리와 통합된 Strands 에이전트를 사용하여 지능형 요리 어시스턴트를 구축하는 방법을 설명합니다. 이전 대화와 개인 취향을 기반으로 개인화된 레스토랑 추천을 제공하기 위해 사용자의 음식 선호도, 식이 제한, 식사 이력에 대한 장기 메모리에 초점을 맞춥니다.

In [ ]:
import logging
import json
from typing import Dict
from datetime import datetime
from botocore.exceptions import ClientError

# 로깅 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("customer-support")

# 필요한 모듈 임포트
from strands import Agent, tool
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent
from ddgs import DDGS
from bedrock_agentcore.memory import MemoryClient

# MemoryClient 초기화
client = MemoryClient(region_name=region_name)

## 13단계: 셀프 매니지드 메모리를 활용한 요리 어시스턴트용 메모리 훅 프로바이더 생성

훅은 에이전트 실행 라이프사이클의 특정 시점에서 실행되는 특수 함수입니다. 커스텀 훅 프로바이더는 셀프 매니지드 메모리 전략을 활용하여 다음과 같이 요리 컨텍스트를 자동으로 관리합니다:

- 셀프 매니지드 메모리 레코드에서 **관련 음식 선호도를 검색**합니다
- 식이 제한, 요리 선호도, 식사 이력에 대한 **맥락 정보를 주입**합니다
- 배치 작업을 사용하여 향후 참조를 위해 **식사 상호작용을 저장**합니다

이를 통해 다음과 같은 원활한 메모리 경험이 만들어집니다:
- 각 쿼리를 처리하기 전에 저장된 음식 선호도를 자동으로 검색합니다
- 식사 이력을 기반으로 컨텍스트 인식 레스토랑 추천을 제공합니다

셀프 매니지드 접근 방식은 음식 선호도가 저장, 검색 및 활용되는 방식을 완전히 제어할 수 있게 하여 식사 추천 경험을 향상시킵니다.

In [ ]:
# 메모리 전략 목록에서 네임스페이스를 가져오는 헬퍼 함수
def get_namespaces(mem_client: MemoryClient, memory_id: str) -> Dict:
    """메모리 전략에 대한 네임스페이스 매핑을 가져옵니다."""
    strategies = mem_client.get_memory_strategies(memory_id)
    return {i["type"]: i["namespaces"][0] for i in strategies}

In [ ]:
class CulinaryAssistantMemoryHooks(HookProvider):
    """요리 어시스턴트 에이전트를 위한 메모리 훅"""
    
    def __init__(self, memory_id: str, namespace: str):
        self.memory_id = memory_id
        self.namespace = namespace
    
    def retrieve_food_preferences(self, event: MessageAddedEvent):
        """식사 쿼리 처리 전 사용자 음식 선호도를 검색합니다"""
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                # 직접 API를 사용하여 음식 선호도를 검색합니다
                response = aws_utils.agentcore_client.retrieve_memory_records(
                    memoryId=self.memory_id,
                    searchCriteria={
                        'searchQuery': user_query,
                        'topK': 5
                    },
                    namespace=self.namespace
                )
                
                memory_records = response.get('memoryRecordSummaries', [])
                
                if memory_records:
                    # 검색된 선호도를 포맷합니다
                    preferences_context = []
                    for record in memory_records:
                        content = record.get('content', {}).get('text', '').strip()
                        if content:
                            preferences_context.append(content)
                    
                    # 음식 선호도를 쿼리에 주입합니다
                    if preferences_context:
                        context_text = "\n".join(preferences_context)
                        original_text = messages[-1]["content"][0]["text"]
                        messages[-1]["content"][0]["text"] = (
                            f"User Food Preferences:\n{context_text}\n\n{original_text}"
                        )
                        logger.info(f"{len(preferences_context)}개의 음식 선호도 레코드를 검색했습니다")
                
            except Exception as e:
                logger.error(f"음식 선호도 검색 실패: {e}")
    
    def save_dining_interaction(self, event: AfterInvocationEvent):
        """에이전트 응답 후 식사 추천 상호작용을 저장합니다"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # 마지막 사용자 쿼리와 에이전트 응답을 가져옵니다
                user_query = None
                agent_response = None
                
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_query and "toolResult" not in msg["content"][0]:
                        user_query = msg["content"][0]["text"]
                        break
                
                if user_query and agent_response:
                    # 직접 API를 사용하여 상호작용을 저장합니다
                    interaction_content = f"Query: {user_query}\nRecommendation: {agent_response}"
                    
                    # 여기서 create_memory_record API를 사용합니다
                    # aws_utils.agentcore_client.create_memory_record(...)
                    
                    logger.info("식사 상호작용이 메모리에 저장되었습니다")
                    
        except Exception as e:
            logger.error(f"식사 상호작용 저장 실패: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """요리 어시스턴트 메모리 훅을 등록합니다"""
        registry.add_callback(MessageAddedEvent, self.retrieve_food_preferences)
        registry.add_callback(AfterInvocationEvent, self.save_dining_interaction)
        logger.info("요리 어시스턴트 메모리 훅이 등록되었습니다")

## 14단계: 요리 어시스턴트 에이전트 생성

In [ ]:
# 요리 어시스턴트용 메모리 훅 생성
print(memory_id)
culinary_hooks = CulinaryAssistantMemoryHooks(memory_id, namespace)

# 요리 어시스턴트 에이전트 생성
culinary_agent = Agent(
    hooks=[culinary_hooks],
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[],  # 필요에 따라 이 도구들을 업데이트하세요
    state={"actor_id": actor_id, "session_id": session_id},
    system_prompt="""You are the Culinary Assistant, a sophisticated restaurant recommendation assistant.

PURPOSE:
- Help users discover restaurants based on their preferences
- Remember user preferences throughout the conversation
- Provide personalized dining recommendations

You have access to a Memory tool that enables you to:
- Store user preferences (dietary restrictions, favorite cuisines, budget preferences, etc.)
- Retrieve previously stored information to personalize recommendations"""
)

print("요리 어시스턴트 에이전트가 메모리 기능과 함께 생성되었습니다")

#### 에이전트가 준비되었습니다.

### 요리 어시스턴트 시나리오를 테스트해 봅시다

In [ ]:
# 저녁 식사 음식 선택에 대해 물어봅니다
response1 = culinary_agent("what are the food choices for Dinner?")
print(f"지원 에이전트: {response1}")

## 15단계: 리소스 정리

불필요한 비용이 발생하지 않도록 생성한 모든 리소스를 정리합니다.

In [ ]:
# 모든 리소스 정리
import importlib
import aws_utils
importlib.reload(aws_utils)

# 업데이트된 코드로 AWSUtils의 새 인스턴스를 생성합니다
aws_utils = aws_utils.AWSUtils(region_name=region_name)

# 자동 검색으로 리소스를 정리합니다
aws_utils.cleanup_resources(discover_resources=True)
print("모든 리소스가 정리되었습니다!")

## 요약

이 노트북에서 다음을 시연했습니다:

1. 셀프 매니지드 메모리에 필요한 AWS 인프라 설정
2. 셀프 매니지드 전략으로 AgentCore 메모리 생성
3. 메모리 처리를 위한 트리거 조건 설정
4. Lambda 기반 메모리 처리 파이프라인 구현
5. 샘플 대화로 메모리 시스템 테스트
6. 추출된 메모리 검색
7. 셀프 매니지드 메모리를 테스트하기 위한 요리 에이전트 생성
8. 모든 리소스 정리

셀프 매니지드 메모리 전략은 메모리 추출에 대한 완전한 제어를 제공하여, 특정 사용 사례에 맞는 커스텀 파이프라인을 구축할 수 있게 합니다.